### Stock Market (Open, Close, High, Low prices over 5 days)

In [2]:
import numpy as np

In [3]:
data = np.array([
    [101.2, 103.5, 104.0, 100.8],
    [103.5, 102.0, 105.2, 101.5],
    [102.0, 106.3, 107.0, 101.9],
    [106.3, 104.8, 108.0, 104.1],
    [104.8, 107.5, 108.5, 104.5]
])

In [4]:
data

array([[101.2, 103.5, 104. , 100.8],
       [103.5, 102. , 105.2, 101.5],
       [102. , 106.3, 107. , 101.9],
       [106.3, 104.8, 108. , 104.1],
       [104.8, 107.5, 108.5, 104.5]])

#### 1. Compute daily return = (close - open) / open for each day

In [5]:
#Extract Open and Close columns
open_prices = data[:, 0]
close_prices = data[:, 1] 

In [6]:
# Calculate daily return
daily_returns = (close_prices - open_prices) / open_prices
print("Daily Returns:", daily_returns)

Daily Returns: [ 0.02272727 -0.01449275  0.04215686 -0.01411101  0.02576336]


#### 2. Compute volatility as std of close prices

In [7]:
# Standard deviation of closing prices
volatility = np.std(close_prices)
print("Close Price Volatility:", volatility)

Close Price Volatility: 1.9528440797974627


#### 3.Compute high-low range per day and identify most volatile day

In [8]:
high_prices = data[:, 2]
low_prices = data[:, 3]

# High-Low range per day
hl_range = high_prices - low_prices

# Most volatile day (0-indexed day count)
most_volatile_day = np.argmax(hl_range)

print("Daily High-Low Ranges:", hl_range)
print("Most Volatile Day (0-indexed):", most_volatile_day)

Daily High-Low Ranges: [3.2 3.7 5.1 3.9 4. ]
Most Volatile Day (0-indexed): 2


#### 4. Compute correlation between open and close prices

In [9]:
# Pearson correlation matrix between Open and Close
corr_matrix = np.corrcoef(open_prices, close_prices)
open_close_corr = corr_matrix[0, 1]

print("Correlation between Open and Close:", open_close_corr)

Correlation between Open and Close: 0.2353613473828931


#### 5. Conceptual: Explain why returns are preferable to prices for modeling

 * Stationarity: Raw stock prices generally exhibit non-stationary trends over time, whereas price returns fluctuate around a stable mean, making statistical modeling and forecasting significantly easier.
 * Scale Independence: Returns normalize asset values into relative percentage changes, allowing direct performance comparison across assets with vastly different price ranges (e.g., a \$10 stock vs. a \$1000 stock).

#### 6. Compute rolling mean (window=3) of close prices

In [10]:
#Slicing / Manual Window Mean
rolling_mean_slice = np.array([
    np.mean(close_prices[i:i+3]) for i in range(len(close_prices) - 2)
])

In [11]:
print("3-Day Rolling Mean (Slice):", rolling_mean_slice)

3-Day Rolling Mean (Slice): [103.93333333 104.36666667 106.2       ]


#### 7. Detect any days with gap up/down larger than 2%

In [12]:
# Gap calculation: (Day_i Open - Day_{i-1} Close) / Day_{i-1} Close
gap_pct = (open_prices[1:] - close_prices[:-1]) / close_prices[:-1]

# Vectorized condition for absolute gaps > 2% (0.02)
large_gaps = np.abs(gap_pct) > 0.02

# Days with gaps > 2% (Day index relative to Day 1 onwards)
gap_days = np.where(large_gaps)[0] + 1

print("Percentage Gaps:", gap_pct)
print("Days with >2% gap (1-indexed):", gap_days)

Percentage Gaps: [0. 0. 0. 0.]
Days with >2% gap (1-indexed): []


#### 8. Calculate average intraday range (high-low) across days

In [13]:
avg_intraday_range = np.mean(high_prices - low_prices)
print("Average Intraday Range:", avg_intraday_range)

Average Intraday Range: 3.9800000000000013


#### 9. Use percent change to evaluate momentum and classify trend direction

In [14]:
# Momentum evaluated via daily percentage returns
momentum = daily_returns

# Classify trend direction: 1 for Positive (Up), -1 for Negative (Down), 0 for Flat
trend_direction = np.where(momentum > 0, 'Up', np.where(momentum < 0, 'Down', 'Flat'))

for i, (m, t) in enumerate(zip(momentum, trend_direction)):
    print(f"Day {i+1}: Return = {m:.4f} ({m*100:.2f}%) -> Trend: {t}")

Day 1: Return = 0.0227 (2.27%) -> Trend: Up
Day 2: Return = -0.0145 (-1.45%) -> Trend: Down
Day 3: Return = 0.0422 (4.22%) -> Trend: Up
Day 4: Return = -0.0141 (-1.41%) -> Trend: Down
Day 5: Return = 0.0258 (2.58%) -> Trend: Up


#### 10. Discuss why price series may not be normally distributed and consequences for models

 * Fat Tails (Kurtosis): Real-world financial returns experience extreme market shocks or crashes far more frequently than predicted by a standard Gaussian bell curve.
 * Skewness: Markets often exhibit asymmetric volatility, dropping faster during sell-offs than they rise during bull markets.
 * Model Consequences: Assuming normal distribution understates financial risk, underprices tail-risk derivatives (such as options), and underestimates the frequency of portfolio losses.